## Data View

In [ ]:
import pandas as pd
import sys
sys.path.append("/home/ec2-user/studies/HUIFA-RNA-SEQ-new/phenoAI/")

from phenokit.preprocessing.qc import MissingValueAnalyzer
from phenokit.preprocessing.io import read_parquet

In [ ]:
pfile = '/home/ec2-user/studies/hpp_datasets/anthropometrics/anthropometrics.parquet'
df_anthro = read_parquet(pfile)
print(df_anthro.shape)
df_anthro.head()

In [ ]:
missing_analyzer = MissingValueAnalyzer(df_anthro)
missing_analyzer.overall_missing_rate()
missing_analyzer.report_features()
missing_analyzer.report_samples()
missing_analyzer.report_heatmap()

# Missing Value Handling

In [ ]:
metadata_col = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_date', 'collection_timestamp', 'timezone']
feature_col = ['bmi', 'height', 'weight', 'waist_circumference', 'hip_circumference', 'waist_to_hip_ratio']
metadata_col = [col for col in metadata_col if col in df_anthro.columns]
df_anthro_metadata = df_anthro[metadata_col]
df_anthro_features = df_anthro[feature_col]

In [ ]:
df_anthro_features_columns = df_anthro_features.columns.tolist()
df_anthro_features.columns = [str(i) for i in range(len(df_anthro_features.columns))]


from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)
imputed_data = imputer.fit_transform(df_anthro_features)
df_anthro_imputed = pd.DataFrame(imputed_data, columns=df_anthro_features_columns)

# Save

In [ ]:
df_anthro_metadata.reset_index(drop=True, inplace=True)
df_anthro_imputed.reset_index(drop=True, inplace=True)
df_anthro = pd.concat([df_anthro_metadata, df_anthro_imputed], axis=1)
df_anthro.to_parquet('pheno_data/202512/002-anthropometrics_imputed.parquet')

In [ ]:
df_bmi = df_anthro[df_anthro['bmi'].notnull()]
df_bmi = df_bmi.reset_index(drop=True)
bmi_min = df_bmi['bmi'].min()
bmi_max = df_bmi['bmi'].max()
bmi_mean = df_bmi['bmi'].mean()
bmi_std = df_bmi['bmi'].std()
print(f"\nBMI - Min: {bmi_min}, Max: {bmi_max}, Mean: {bmi_mean:.2f}, Std: {bmi_std:.2f}")